In [7]:
# -*- coding: utf-8 -*-
"""experimetns.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1MJVgMpWIoZxGZEu9e8Tgk4jWruM-NMN5
"""

# !cd /content/drive/MyDrive && zip -0 -r -q synth_final.zip synth_extracted_final labels_runic.csv

# Commented out IPython magic to ensure Python compatibility.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from __future__ import annotations

import os
import re
import sys
import json
import glob
import math
import shutil
import zipfile
import random
import argparse
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Callable

import numpy as np
import pandas as pd
import torch

In [8]:
pip install -U bitsandbytes>=0.46.1 qwen_vl_utils

Note: you may need to restart the kernel to use updated packages.


In [9]:
from huggingface_hub import login
from transformers import TrainerCallback
from huggingface_hub import upload_folder
from pathlib import Path


login("YOUR_HF_TOKEN")

from huggingface_hub import HfApi

api = HfApi()

print(
    api.whoami()
)

HF_REPO = "AntoniusPerf/trocr-checkpoints"


class HFCheckpointCallback(TrainerCallback):

    def on_save(self, args, state, control, **kwargs):

        model_name = Path(args.output_dir).name

        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"

        if not ckpt.exists():
            return control

        print(f"\nUploading {ckpt} ...")

        upload_folder(
            repo_id=HF_REPO,
            folder_path=str(ckpt),
            path_in_repo=f"{model_name}/checkpoint-{state.global_step}",
            commit_message=f"{model_name} step {state.global_step}"
        )

        print("Upload finished")

        return control

{'type': 'user', 'id': '6713ef710d49e1c97f8817f5', 'name': 'AntoniusPerf', 'fullname': 'Perfilev', 'email': '<email>', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': '/avatars/0c9d6ead1ba8d091b112230581ba8416.svg', 'orgs': [{'type': 'org', 'id': '6808f41dcb9a2da1daa0b54f', 'name': 'HSE-Chukchi-NLP', 'fullname': 'HSE Chukchi NLP', 'email': None, 'canPay': False, 'billingMode': 'postpaid', 'periodEnd': None, 'avatarUrl': 'https://www.gravatar.com/avatar/cee61e0d617a9bb200825a7d43c7a52b?d=retro&size=100', 'roleInOrg': 'write'}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'kaggle_token', 'role': 'write', 'createdAt': '2026-06-03T05:28:01.429Z'}}}


In [10]:
# =============================================================================
# 0. ГЛОБАЛЬНАЯ КОНФИГУРАЦИЯ
# =============================================================================

@dataclass
class GlobalConfig:
    # --- источники данных ---
    # Kaggle: датасеты уже смонтированы как папки в /kaggle/input/...
    SYNTH_ZIP: str = "/kaggle/input/datasets/zhopa228/synth-final/synth_extracted_final"
    SYNTH_ROOT: str = "/kaggle/working/synth_local"            # куда распаковывать синтетику (если это zip)
    GOLD_SOURCE: str = "/kaggle/input/datasets/zhopa228/val-dataset/val_dataset"
    GOLD_ROOT: str = "/kaggle/working/real"
    GOLD_CSV_NAME: str = "real_corpus.csv"              # внутри gold-источника (если имя известно)
    OUT_ROOT: str = "/kaggle/working/runic_runs"        # корень для чекпойнтов/результатов
    ARCHIVE_ROOT: str = "/kaggle/working/runic_archives" # zip-архивы runs / best

    # --- сплиты синтетики (группировка по уникальному слову, без утечки) ---
    VAL_FRAC: float = 0.05
    SEED: int = 1003

    # --- канонические сплиты/метки из CSV билдера (labels_runic.csv) ---
    # Если путь задан — метки И сплиты берутся из CSV (твой builder), а не из имён
    # файлов; ожидаемые столбцы: filename, transliteration, runic, split[, mode].
    SYNTH_LABELS_CSV: str = ""                 # "" → разметка из имён файлов (как было)
    SYNTH_EVAL_SPLIT: str = "val"              # синт. held-out для CER (val | test)
    EXCLUDE_GOLD_LEXICON: bool = False          # убрать из синтетики слова, входящие в gold set

    # --- протокол оценки ---
    BOOTSTRAP_B: int = 2000          # число бутстрэп-выборок (в тексте \tofill{B})
    BOOTSTRAP_SEED: int = 0
    SYNTH_VAL_EVAL_MAX: int = 300    # сколько синт-val примеров брать на CER (генерация дорогая)
    EARLY_STOPPING_PATIENCE: int = 3  # остановка если N eval-раундов без улучшения (0 → отключён)

    # --- постобработка для метрик (единый протокол для ВСЕХ моделей) ---
    STRIP_WHITESPACE_FOR_CER: bool = True   # убрать пробелы (артефакт декодинга Qwen)
    STRIP_SEPARATORS_FOR_CER: bool = True   # CER считается по графемам, без ·/:/+
    WORD_SEP_CHARS: str = " ·:+/"           # разделители слов для WER

    # --- абляция: режимы синтеза определяются ИМЕНЕМ ПОДПАПКИ в SYNTH_ROOT ---
    # ожидается раскладка вида:
    #   synth_local/inpainting/*.png
    #   synth_local/controlnet/*.png
    #   (для «смеси» используются обе папки)
    ABLATION_MODES: tuple = ("inpainting", "controlnet")


CFG = GlobalConfig()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def zip_dir(src_dir: str, zip_path: str) -> str:
    """Упаковать папку в zip. Возвращает путь к архиву."""
    src = Path(src_dir)
    dst = Path(zip_path)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        dst.unlink()
    base_name = dst.with_suffix('')
    shutil.make_archive(str(base_name), 'zip', root_dir=str(src))
    return str(dst)


def archive_run_outputs(run_name: str, best_subdir: str = "best") -> tuple[str, str]:
    """Сохранить полный run и best-чекпойнт в отдельные zip-архивы."""
    run_dir = Path(CFG.OUT_ROOT) / run_name
    archive_root = Path(CFG.ARCHIVE_ROOT)
    run_zip = archive_root / "runs" / f"{run_name}.zip"
    best_dir = run_dir / best_subdir
    best_zip = archive_root / "best" / f"{run_name}_best.zip"

    if run_dir.exists():
        zip_dir(str(run_dir), str(run_zip))
        print(f"[zip] run → {run_zip}")
    else:
        print(f"[zip] run dir not found: {run_dir}")

    if best_dir.exists():
        zip_dir(str(best_dir), str(best_zip))
        print(f"[zip] best → {best_zip}")
    else:
        print(f"[zip] best dir not found: {best_dir}")

    return str(run_zip), str(best_zip)

In [11]:
# =============================================================================
# 1. ТРАНСЛИТЕРАЦИЯ (согласовано с runic_transliteration.py, конвенция Rundata)
# =============================================================================
# Карты дублируются здесь, чтобы стенд был самодостаточным и запускался на
# Kaggle (модуль проекта импортирует google.colab и вне Colab не загружается).

ELDER_MAP = {
    "ᚠ": "f", "ᚢ": "u", "ᚦ": "þ", "ᚨ": "a", "ᚱ": "r", "ᚲ": "k", "ᚷ": "g", "ᚹ": "w",
    "ᚺ": "h", "ᚾ": "n", "ᛁ": "i", "ᛃ": "j", "ᛇ": "ï", "ᛈ": "p", "ᛉ": "R", "ᛊ": "s",
    "ᛏ": "t", "ᛒ": "b", "ᛖ": "e", "ᛗ": "m", "ᛚ": "l", "ᛜ": "ŋ", "ᛞ": "d", "ᛟ": "o",
}
YOUNGER_MAP = {
    "ᚠ": "f", "ᚢ": "u", "ᚦ": "þ", "ᚬ": "ą", "ᚱ": "r", "ᚴ": "k", "ᚼ": "h", "ᚾ": "n",
    "ᛁ": "i", "ᛅ": "a", "ᛋ": "s", "ᛏ": "t", "ᛒ": "b", "ᛘ": "m", "ᛚ": "l", "ᛦ": "ʀ",
}
FUTHORC_MAP = {
    "ᚠ": "f", "ᚢ": "u", "ᚦ": "þ", "ᚩ": "o", "ᚱ": "r", "ᚳ": "c", "ᚷ": "g", "ᚹ": "w",
    "ᚻ": "h", "ᚾ": "n", "ᛁ": "i", "ᛄ": "j", "ᛇ": "eo", "ᛈ": "p", "ᛉ": "x", "ᛋ": "s",
    "ᛏ": "t", "ᛒ": "b", "ᛖ": "e", "ᛗ": "m", "ᛚ": "l", "ᛝ": "ng", "ᛞ": "d", "ᚪ": "a",
    "ᚫ": "æ", "ᚣ": "y", "ᛠ": "ea", "ᛡ": "ia", "ᛣ": "q", "ᛤ": "k", "ᛥ": "st",
}
# Разделители слов рунической пунктуации
PUNCT_MAP = {"᛫": "·", "᛬": ":", "᛭": "+"}

# Объединённая карта; Elder перезаписывает конфликты (как в проекте)
COMBINED_MAP = {**YOUNGER_MAP, **FUTHORC_MAP, **ELDER_MAP, **PUNCT_MAP}

# Любой кодпойнт рунического блока Unicode
RUNE_RX = re.compile(r"[\u16A0-\u16FF]+")
# Имя синт-файла: поддержка обоих форматов
#   syn_w0_000006_fargaiR_mik_ybi_ᚠᛅᚱᚵᛅᛁᛦ.png  (с batch-ID)
#   syn_000006_fargaiR_mik_ybi_ᚠᛅᚱᚵᛅᛁᛦ.png     (без batch-ID)
_RUNIC_CHAR_RX = re.compile(r"[\u16A0-\u16FF]")


def _parse_synth_fname(name: str) -> Optional[dict]:
    """Разобрать имя синтетического файла → dict(translit, runic) | None."""
    stem = Path(name).stem
    m = _RUNIC_CHAR_RX.search(stem)
    if not m:
        return None
    pre = stem[: m.start()]
    if not pre.endswith("_"):
        return None
    prefix = pre[:-1]               # без завершающего '_'
    runic = stem[m.start():]
    parts = prefix.split("_", 3)    # syn, [batch], idx, translit...
    if len(parts) < 3 or parts[0] != "syn":
        return None
    try:
        int(parts[1])               # syn_000006_translit… → нет batch-ID
        translit_raw = "_".join(parts[2:])
    except ValueError:              # syn_w0_000006_translit… → есть batch-ID
        if len(parts) < 4:
            return None
        try:
            int(parts[2])
        except ValueError:
            return None
        translit_raw = parts[3]
    return {"translit": translit_raw.replace("_", " "), "runic": runic}


def transliterate(runic: str, mapping: dict = COMBINED_MAP, unknown: str = "?") -> str:
    """Руническая строка → латинская транслитерация (конвенция Rundata)."""
    return "".join(mapping.get(ch, PUNCT_MAP.get(ch, unknown)) for ch in runic)

In [12]:
# =============================================================================
# 2. ПОДГОТОВКА ДАННЫХ
# =============================================================================

def unzip_if_needed(src: str, dst: str) -> str:
    """Распаковать zip в dst (или вернуть путь к уже готовой папке)."""
    p = Path(src)
    if p.is_dir():
        return str(p)
    Path(dst).mkdir(parents=True, exist_ok=True)
    has_png = Path(dst).exists() and any(Path(dst).rglob("*.png"))
    if not has_png:
        for z in glob.glob(src):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dst)
        print(f"распаковано: {src} → {dst}")
    else:
        print(f"уже распаковано: {dst}")
    return dst


def _infer_synth_mode(path: Path, root: Path) -> Optional[str]:
    """Определить режим синтеза по имени подпапки (для абляции)."""
    try:
        rel_parts = path.relative_to(root).parts
    except ValueError:
        rel_parts = path.parts
    for part in rel_parts:
        for m in CFG.ABLATION_MODES:
            if part.lower() == m.lower():
                return m
    return None


def _index_pngs(images_root: str) -> dict:
    """filename → полный путь (рекурсивно, включая подпапки режимов)."""
    return {p.name: str(p) for p in Path(images_root).rglob("*.png")}


def load_synth_labels_csv(csv_path: str, images_root: str,
                          mapping: dict = COMBINED_MAP) -> pd.DataFrame:
    """
    Загрузить разметку синтетики из CSV билдера (labels_runic.csv).

    Метки берутся напрямую из CSV (исходная латиница с пробелами-разделителями
    слов), а не восстанавливаются по карте транслитерации из имени файла — это
    и точнее (нет '?' для нестандартных рун), и делает WER осмысленной.

    Ожидаемые столбцы: filename, transliteration, runic, split.
    Опционально: mode/regime (для абляции). Путь к изображению восстанавливается
    сопоставлением filename ↔ PNG в images_root (поэтому CSV переносим отдельно
    от картинок или внутри того же архива).
    """
    p = Path(csv_path)
    if not p.exists():                      # не нашли по пути — ищем по имени в распакованном
        hits = glob.glob(f"{images_root}/**/{Path(csv_path).name}", recursive=True)
        assert hits, f"CSV меток не найден: ни {csv_path}, ни {Path(csv_path).name} в {images_root}"
        p = Path(hits[0])
    df = pd.read_csv(p)

    # имя файла
    fcol = ("filename" if "filename" in df.columns
            else "file_name" if "file_name" in df.columns else None)
    assert fcol, f"в {p} нет столбца filename/file_name"
    df = df.rename(columns={fcol: "filename"})

    # целевая транслитерация
    if "translit" not in df.columns:
        tcol = "transliteration" if "transliteration" in df.columns else None
        assert tcol, f"в {p} нет столбца transliteration/translit"
        df["translit"] = df[tcol].astype(str)
    else:
        df["translit"] = df["translit"].astype(str)

    # руны (если нет — восстановим из имени; нужны для контроля утечки по слову)
    if "runic" not in df.columns:
        df["runic"] = df["filename"].map(
            lambda n: (_parse_synth_fname(str(n)) or {}).get("runic", ""))

    # сплиты
    if "split" in df.columns:
        df["split"] = df["split"].astype(str).str.lower()
    else:
        print("[!] в CSV нет столбца split — сплиты построит стенд (make_splits).")

    # режим синтеза: из столбца mode/regime, иначе попытка вывести из пути (подпапки)
    if "mode" not in df.columns:
        rcol = "regime" if "regime" in df.columns else None
        df["mode"] = df[rcol] if rcol else None

    # путь к PNG
    idx = _index_pngs(images_root)
    df["path"] = df["filename"].map(idx.get)
    if df["mode"].isna().all():             # режим не задан столбцом → пробуем из пути
        root = Path(images_root)
        df["mode"] = df["path"].map(
            lambda s: _infer_synth_mode(Path(s), root) if isinstance(s, str) else None)

    miss = int(df["path"].isna().sum())
    if miss:
        print(f"[!] нет PNG для {miss} строк CSV — пропущены")
        df = df[df["path"].notna()].reset_index(drop=True)

    keep = ["filename", "path", "runic", "translit", "split", "mode"]
    df = df[[c for c in keep if c in df.columns]].reset_index(drop=True)
    dist = df["split"].value_counts().to_dict() if "split" in df.columns else "—"
    print(f"синтетика из CSV: строк={len(df)} | csv={p} | сплиты={dist}")
    return df


def parse_synth_labels(images_root: str, mapping: dict = COMBINED_MAP) -> pd.DataFrame:
    """
    Построить разметку синтетики.

    Если задан CFG.SYNTH_LABELS_CSV — метки и сплиты читаются из CSV билдера
    (см. load_synth_labels_csv). Иначе — восстанавливаются из имён файлов.

    Возвращает DataFrame со столбцами:
        filename, path, runic, translit, mode[, split (при чтении из CSV)].
    """
    if CFG.SYNTH_LABELS_CSV:
        return load_synth_labels_csv(CFG.SYNTH_LABELS_CSV, images_root, mapping)

    root = Path(images_root)
    rows = []
    for path in root.rglob("*.png"):
        parsed = _parse_synth_fname(path.name)
        if parsed is None:
            continue
        rows.append({
            "filename": path.name,
            "path": str(path),
            "runic": parsed["runic"],
            "translit": parsed["translit"],
            "mode": _infer_synth_mode(path, root),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        print(f"[!] 0 файлов распознано в {images_root}. Примеры имён:")
        for p in sorted(Path(images_root).rglob("*.png"))[:5]:
            print(f"    {p.name}")
    return df


def make_splits(df: pd.DataFrame, gold_lexicon: set,
                val_frac: float = CFG.VAL_FRAC, seed: int = CFG.SEED,
                respect_existing: Optional[bool] = None) -> pd.DataFrame:
    """
    Разбить синтетику на train/val с группировкой по уникальному слову
    (лексическая непересекаемость) и исключением лексикона gold set (контроль
    утечки).

    Если в df уже есть столбец split И включён канонический режим
    (CFG.SYNTH_LABELS_CSV) — сплиты НЕ пересчитываются, берутся как есть
    (train/val/test из CSV билдера). respect_existing=True/False форсирует выбор.
    """
    df = df.copy()
    # исключаем слова, попавшие в gold set
    n_before = len(df)
    if gold_lexicon:
        df = df[~df["runic"].isin(gold_lexicon)].reset_index(drop=True)
        print(f"исключено по лексикону gold set: {n_before - len(df)} строк")

    use_existing = (respect_existing if respect_existing is not None
                    else ("split" in df.columns and bool(CFG.SYNTH_LABELS_CSV)))
    if use_existing and "split" in df.columns:
        print(f"синтетика: КАНОНИЧЕСКИЕ сплиты из CSV | "
              f"строк={len(df)} | {df.split.value_counts().to_dict()}")
        return df

    units = df["runic"].drop_duplicates().sample(frac=1.0, random_state=seed).tolist()
    nv = max(1, int(len(units) * val_frac))
    val_u = set(units[:nv])
    df["split"] = df["runic"].map(lambda v: "val" if v in val_u else "train")
    print(f"синтетика: строк={len(df)} | сплиты={df.split.value_counts().to_dict()}")
    return df


# Псевдонимы возможных имён столбцов-срезов в gold set
_SLICE_ALIASES = {
    "system": ["system", "futhark", "script", "alphabet", "система"],
    "material": ["material", "support", "carrier", "материал"],
    "preservation": ["preservation", "condition", "сохранность"],
}


def load_gold_set(source: str = None, csv_name: str = None) -> pd.DataFrame:
    """
    Загрузить реальный gold set: фотографии + CSV с транслитерацией.

    CSV обязан содержать столбцы filename и translit. Опционально —
    столбцы срезов гетерогенности (система/материал/сохранность).
    Возвращает DataFrame: file_name, path, gt, [system, material, preservation].
    """
    source = source or CFG.GOLD_SOURCE
    csv_name = csv_name or CFG.GOLD_CSV_NAME
    root = unzip_if_needed(source, CFG.GOLD_ROOT)

    if csv_name:
        csvs = glob.glob(f"{root}/**/{csv_name}", recursive=True)
    else:
        csvs = glob.glob(f"{root}/**/*.csv", recursive=True)

    assert csvs, f"CSV с gold set не найден в {root}"
    df = pd.read_csv(sorted(csvs)[0])
    fcol = "filename" if "filename" in df.columns else "file_name"
    df = df.rename(columns={fcol: "file_name"})
    df["gt"] = df["translit"].astype(str)
    index = {p.name: str(p) for p in Path(root).rglob("*.png")}
    df["path"] = df["file_name"].map(index.get)
    missing = df["path"].isna().sum()
    if missing:
        print(f"[!] нет фото для {missing} строк gold set — они будут пропущены")
        df = df[df["path"].notna()].reset_index(drop=True)
    # нормализуем имена столбцов-срезов
    lower = {c.lower(): c for c in df.columns}
    for canon, aliases in _SLICE_ALIASES.items():
        for a in aliases:
            if a in lower and canon not in df.columns:
                df[canon] = df[lower[a]]
                break
    print(f"gold set: строк={len(df)} | csv={csvs[0]}")
    return df

In [13]:
# =============================================================================
# 3. НОРМАЛИЗАЦИЯ И МЕТРИКИ (единый протокол для всех моделей)
# =============================================================================

_WS_RX = re.compile(r"\s+")


def norm_cer(s: str) -> str:
    """Нормализация строки для CER/NED/SeqAcc и анализа ошибок."""
    s = (s or "").strip()
    if CFG.STRIP_WHITESPACE_FOR_CER:
        s = _WS_RX.sub("", s)
    else:
        s = _WS_RX.sub(" ", s)
    if CFG.STRIP_SEPARATORS_FOR_CER:
        for ch in "·:+":
            s = s.replace(ch, "")
    return s


def word_tokens(s: str) -> list:
    """Токенизация на слова для WER (по рунической точке-разделителю/пробелу).

    ВНИМАНИЕ: WER информативна только если в эталоне реально присутствуют
    разделители слов. Если транслитерация слитная (как в части синтетики),
    WER вырождается в посегментную метрику ≈ (1 − SequenceAccuracy).
    """
    s = (s or "").strip()
    s = _WS_RX.sub(" ", s)
    pat = "[" + re.escape(CFG.WORD_SEP_CHARS) + "]+"
    toks = [t for t in re.split(pat, s) if t]
    return toks


def edit_distance(a, b) -> int:
    """Расстояние Левенштейна для любых последовательностей (строки/списки)."""
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        ai = a[i - 1]
        for j in range(1, lb + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[lb]


def compute_metrics(refs: list, preds: list) -> dict:
    """
    Посчитать CER, WER, NED, SequenceAccuracy + покомпонентные массивы
    для бутстрэпа. Нормализация применяется одинаково к refs и preds.
    """
    refs_c = [norm_cer(r) for r in refs]
    preds_c = [norm_cer(p) for p in preds]

    char_dist, char_len = [], []
    ned, seq_ok = [], []
    word_dist, word_len = [], []
    for r, p in zip(refs_c, preds_c):
        d = edit_distance(p, r)
        char_dist.append(d)
        char_len.append(max(len(r), 1))
        ned.append(d / max(len(r), 1) if len(r) else (0.0 if len(p) == 0 else 1.0))
        seq_ok.append(1.0 if p == r else 0.0)
    for r, p in zip(refs, preds):
        rw, pw = word_tokens(r), word_tokens(p)
        word_dist.append(edit_distance(pw, rw))
        word_len.append(max(len(rw), 1))

    cer = float(np.sum(char_dist) / max(np.sum(char_len), 1))
    wer = float(np.sum(word_dist) / max(np.sum(word_len), 1))
    return {
        "CER": cer, "WER": wer,
        "NED": float(np.mean(ned)), "SeqAcc": float(np.mean(seq_ok)),
        "n": len(refs),
        "_char_dist": np.asarray(char_dist), "_char_len": np.asarray(char_len),
    }


def bootstrap_cer(char_dist: np.ndarray, char_len: np.ndarray,
                  B: int = None, seed: int = None) -> tuple:
    """Бутстрэп-оценка CER: (точечная, медиана, ДИ95 низ, ДИ95 верх)."""
    B = B or CFG.BOOTSTRAP_B
    seed = CFG.BOOTSTRAP_SEED if seed is None else seed
    rng = np.random.default_rng(seed)
    n = len(char_dist)
    point = float(char_dist.sum() / max(char_len.sum(), 1))
    vals = np.empty(B)
    for k in range(B):
        idx = rng.integers(0, n, n)
        vals[k] = char_dist[idx].sum() / max(char_len[idx].sum(), 1)
    return point, float(np.median(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def paired_bootstrap_delta(distA, lenA, distB, lenB, B: int = None, seed: int = None) -> dict:
    """
    Парный бутстрэп разности CER (модель A − модель B) на ОДНИХ И ТЕХ ЖЕ
    надписях gold set (согласованные ресэмплы). Значимо, если ДИ не содержит 0.
    """
    B = B or CFG.BOOTSTRAP_B
    seed = CFG.BOOTSTRAP_SEED if seed is None else seed
    rng = np.random.default_rng(seed)
    distA, lenA = np.asarray(distA), np.asarray(lenA)
    distB, lenB = np.asarray(distB), np.asarray(lenB)
    n = len(distA)
    deltas = np.empty(B)
    for k in range(B):
        idx = rng.integers(0, n, n)
        cerA = distA[idx].sum() / max(lenA[idx].sum(), 1)
        cerB = distB[idx].sum() / max(lenB[idx].sum(), 1)
        deltas[k] = cerA - cerB
    lo, hi = float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))
    med = float(np.median(deltas))
    return {"delta_median": med, "ci_low": lo, "ci_high": hi,
            "significant": (lo > 0) or (hi < 0)}


def error_analysis(refs: list, preds: list, topk: int = 15) -> dict:
    """
    Типология ошибок: доли S/D/I, частотные пары подстановок (предск.→эталон),
    матрица ошибок графем. Требует rapidfuzz.
    """
    from collections import Counter, defaultdict
    from rapidfuzz.distance import Levenshtein

    refs_c = [norm_cer(r) for r in refs]
    preds_c = [norm_cer(p) for p in preds]

    S = D = I = 0
    sub_pairs = Counter()                       # (pred_char, ref_char) → n
    conf = defaultdict(Counter)                 # true_char → Counter(pred_char)

    for r, p in zip(refs_c, preds_c):
        for op in Levenshtein.editops(r, p):    # преобразование r → p
            if op.tag == "replace":
                S += 1
                sub_pairs[(p[op.dest_pos], r[op.src_pos])] += 1
            elif op.tag == "delete":
                D += 1                           # руна эталона пропущена
            elif op.tag == "insert":
                I += 1                           # лишняя руна в предсказании
        # диагональ матрицы (верно распознанные) — из equal-блоков
        for blk in Levenshtein.opcodes(r, p):
            if blk.tag == "equal":
                for kk in range(blk.src_end - blk.src_start):
                    c = r[blk.src_start + kk]
                    conf[c][c] += 1
            elif blk.tag == "replace":
                nn = min(blk.src_end - blk.src_start, blk.dest_end - blk.dest_start)
                for kk in range(nn):
                    conf[r[blk.src_start + kk]][p[blk.dest_start + kk]] += 1

    tot = max(S + D + I, 1)
    return {
        "S": S, "D": D, "I": I,
        "S_frac": S / tot, "D_frac": D / tot, "I_frac": I / tot,
        "top_subs": sub_pairs.most_common(topk),
        "confusion": conf,
    }


def save_confusion(conf: dict, path: str) -> None:
    """Сохранить нормированную по строкам матрицу ошибок графем в CSV (+ PNG)."""
    chars = sorted(set(conf.keys()) | {c for row in conf.values() for c in row})
    mat = pd.DataFrame(0.0, index=chars, columns=chars)
    for t, row in conf.items():
        for p, c in row.items():
            mat.loc[t, p] = c
    mat = mat.div(mat.sum(axis=1).replace(0, 1), axis=0)  # нормировка по строкам (по эталону)
    mat.to_csv(path, encoding="utf-8")
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(max(6, len(chars) * 0.35),) * 2)
        im = ax.imshow(mat.values, aspect="auto")
        ax.set_xticks(range(len(chars))); ax.set_xticklabels(chars, fontsize=7)
        ax.set_yticks(range(len(chars))); ax.set_yticklabels(chars, fontsize=7)
        ax.set_xlabel("предсказано"); ax.set_ylabel("эталон")
        fig.colorbar(im, ax=ax, fraction=0.046)
        fig.tight_layout(); fig.savefig(path.replace(".csv", ".png"), dpi=150)
        plt.close(fig)
    except Exception as e:
        print(f"[i] PNG матрицы не сохранён ({e}); CSV доступен: {path}")

In [14]:
# =============================================================================
# 4. КОНФИГУРАЦИИ МОДЕЛЕЙ (реестр экспериментов)
# =============================================================================

@dataclass
class ModelConfig:
    name: str                       # ключ эксперимента
    family: str                     # "trocr" | "qwen"
    model_id: str                   # идентификатор HuggingFace
    params: str = ""                # для столбца «Парам.» в таблице
    # обучение
    epochs: int = 3
    lr: float = 1e-4
    batch_size: int = 1
    grad_accum: int = 16
    target: str = "translit"
    # QLoRA / LoRA
    use_lora: bool = True
    quantize_4bit: bool = True      # False для TrOCR
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    # qwen-specific
    max_pixels: int = 512 * 512
    gen_max_new_tokens: int = 64
    # trocr-specific
    maxlen: int = 48
    note: str = ""

    @property
    def out_dir(self) -> str:
        return str(Path(CFG.OUT_ROOT) / self.name)


# Реальные идентификаторы (проверены на HuggingFace):
#   Qwen2-VL   : 2B / 7B / 72B            → класс Qwen2VLForConditionalGeneration
#   Qwen2.5-VL : 3B / 7B / 32B / 72B      → класс Qwen2_5_VLForConditionalGeneration
#   Qwen3-VL   : 2B / 4B / 8B / 32B (dense) → класс Qwen3VLForConditionalGeneration
# Все три семейства грузятся через AutoModelForImageTextToText (transformers>=4.57).
EXPERIMENTS = {
    # --- baseline 1: специализированный распознаватель ---
    "trocr-base": ModelConfig(
        name="trocr-base", family="trocr", model_id="microsoft/trocr-base-stage1",
        params="~334 млн", epochs=25, lr=5e-5, batch_size=8, grad_accum=2,
        quantize_4bit=False, maxlen=48,
        note="влезает в любой GPU; LoRA на декодер. При проблемах PEFT+VED → use_lora=False",
    ),
    "trocr-large": ModelConfig(
    name="trocr-large", family="trocr", model_id="microsoft/trocr-large-stage1",
    params="~558 млн", epochs=25, lr=3e-5, batch_size=4, grad_accum=4,
    quantize_4bit=False, maxlen=48,
    note="один T4: ок; LoRA на декодер",
),
    # --- baseline 2: Qwen2-VL ---
    "qwen2vl-2b": ModelConfig(
        name="qwen2vl-2b", family="qwen", model_id="Qwen/Qwen2-VL-2B-Instruct",
        params="~2 млрд", max_pixels=512 * 512,
        note="один T4: ок",
    ),
    "qwen2vl-7b": ModelConfig(
        name="qwen2vl-7b", family="qwen", model_id="Qwen/Qwen2-VL-7B-Instruct",
        params="~7 млрд", max_pixels=384 * 384,
        note="один T4: на грани (OOM-риск); надёжнее Kaggle 2×T4 / Colab A100",
    ),
    # --- baseline 2: Qwen3-VL ---
    "qwen3vl-2b": ModelConfig(
        name="qwen3vl-2b", family="qwen", model_id="Qwen/Qwen3-VL-2B-Instruct",
        params="~2 млрд", max_pixels=512 * 512,
        note="один T4: ок",
    ),
    "qwen3vl-4b": ModelConfig(
        name="qwen3vl-4b", family="qwen", model_id="Qwen/Qwen3-VL-4B-Instruct",
        params="~4.4 млрд", max_pixels=448 * 448,
        note="один T4: ок (4-бит)",
    ),
    "qwen3vl-8b": ModelConfig(
        name="qwen3vl-8b", family="qwen", model_id="Qwen/Qwen3-VL-8B-Instruct",
        params="~8 млрд", max_pixels=384 * 384,
        note="один T4: на грани; надёжнее Kaggle 2×T4 / Colab A100",
    ),
    # --- опционально: Qwen2.5-VL (если в главе «3B/7B» имелась в виду версия 2.5) ---
    "qwen25vl-3b": ModelConfig(
        name="qwen25vl-3b", family="qwen", model_id="Qwen/Qwen2.5-VL-3B-Instruct",
        params="~3 млрд", max_pixels=512 * 512,
        note="один T4: ок",
    ),
    "qwen25vl-7b": ModelConfig(
        name="qwen25vl-7b", family="qwen", model_id="Qwen/Qwen2.5-VL-7B-Instruct",
        params="~7 млрд", max_pixels=384 * 384,
        note="один T4: на грани; надёжнее Kaggle 2×T4 / Colab A100",
    ),
}

In [15]:
# =============================================================================
# 5. TrOCR — обучение (LoRA на декодер) и оценка
# =============================================================================

def _build_trocr(cfg: ModelConfig, for_train: bool = True, ckpt_dir: str = None):
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel

    src = ckpt_dir if (ckpt_dir and not cfg.use_lora) else cfg.model_id
    proc = TrOCRProcessor.from_pretrained(ckpt_dir or cfg.model_id)
    model = VisionEncoderDecoderModel.from_pretrained(src, low_cpu_mem_usage=False)
    model.encoder.pooler = None
    # Seq2SeqTrainer обращается к config.vocab_size; у VED он в decoder.
    # Property на КЛАССЕ (не инстансе) переживает сериализацию чекпойнтов.
    # setter-заглушка нужна, т.к. transformers пишет в config.vocab_size при инициализации.
    _VEDCfg = type(model.config)
    if not isinstance(_VEDCfg.__dict__.get("vocab_size"), property):
        _VEDCfg.vocab_size = property(
            lambda self: self.decoder.vocab_size,
            lambda self, v: None,               # no-op: всегда читаем из decoder
        )

    # фикс meta-тензоров синусоидальных позиционных эмбеддингов (trocr-base-stage1)
    for m in model.modules():
        if hasattr(m, "_float_tensor"):
            m._float_tensor = torch.zeros(1)
        w = getattr(m, "weights", None)
        if torch.is_tensor(w) and w.is_meta:
            n, d = w.shape
            m.weights = m.get_embedding(n, d, getattr(m, "padding_idx", None))

    tok = proc.tokenizer
    model.config.decoder_start_token_id = tok.cls_token_id
    model.config.pad_token_id = tok.pad_token_id
    model.config.eos_token_id = tok.sep_token_id

    g = model.generation_config
    g.max_length = cfg.maxlen
    g.num_beams = 4
    g.length_penalty = 1.0          # было 2.0 — для коротких транслитераций завышало длину
    g.no_repeat_ngram_size = 3
    g.decoder_start_token_id = tok.cls_token_id
    g.pad_token_id = tok.pad_token_id
    g.eos_token_id = tok.sep_token_id

    if cfg.use_lora and for_train:
        from peft import LoraConfig, get_peft_model
        # LoRA на self-/cross-attention декодера (модули Bart-стиля)
        lcfg = LoraConfig(r=cfg.lora_r, lora_alpha=cfg.lora_alpha,
                          lora_dropout=cfg.lora_dropout, bias="none",
                          target_modules=["q_proj", "k_proj", "v_proj", "out_proj"])
        model = get_peft_model(model, lcfg)
        model.print_trainable_parameters()

    model.to("cuda")
    for m in model.modules():       # пост-фикс девайса синусоид
        w = getattr(m, "weights", None)
        if torch.is_tensor(w) and w.device.type != "cuda":
            m.weights = w.to("cuda")
    return model, proc


class _TrOCRDataset(torch.utils.data.Dataset):
    def __init__(self, df, proc, target, maxlen):
        self.df = df.reset_index(drop=True)
        self.proc, self.target, self.maxlen = proc, target, maxlen

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        from PIL import Image
        r = self.df.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        pv = self.proc(images=img, return_tensors="pt").pixel_values.squeeze(0)
        ids = self.proc.tokenizer(str(r[self.target]), padding="max_length",
                                  truncation=True, max_length=self.maxlen).input_ids
        ids = [t if t != self.proc.tokenizer.pad_token_id else -100 for t in ids]
        return {"pixel_values": pv, "labels": torch.tensor(ids)}


def train_trocr(cfg: ModelConfig, df: pd.DataFrame) -> str:
    from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments,
                              default_data_collator, EarlyStoppingCallback)
    set_seed(CFG.SEED)
    model, proc = _build_trocr(cfg, for_train=True)

    def _metrics(p):
        ids = p.label_ids
        ids[ids == -100] = proc.tokenizer.pad_token_id
        preds = proc.batch_decode(p.predictions, skip_special_tokens=True)
        refs = proc.batch_decode(ids, skip_special_tokens=True)
        return {"cer": compute_metrics(refs, preds)["CER"]}

    cbs = []

    if CFG.EARLY_STOPPING_PATIENCE > 0:
        cbs.append(
            EarlyStoppingCallback(
                early_stopping_patience=CFG.EARLY_STOPPING_PATIENCE
            )
        )
    
    cbs.append(HFCheckpointCallback())

    args = Seq2SeqTrainingArguments(
        output_dir=cfg.out_dir, predict_with_generate=True,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False,
        per_device_train_batch_size=cfg.batch_size, per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum, num_train_epochs=cfg.epochs,
        learning_rate=cfg.lr, warmup_ratio=0.05, fp16=True,
        dataloader_num_workers=2, logging_steps=50, save_total_limit=2, report_to="none",
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=_TrOCRDataset(df[df.split == "train"], proc, cfg.target, cfg.maxlen),
        eval_dataset=_TrOCRDataset(df[df.split == "val"], proc, cfg.target, cfg.maxlen),
        data_collator=default_data_collator, compute_metrics=_metrics,
        callbacks=cbs,
    )
    trainer.train()

    best = str(Path(cfg.out_dir) / "best")
    if cfg.use_lora:                # слить адаптер → сохранить полную модель для оценки
        model = model.merge_and_unload()
    model.save_pretrained(best)
    proc.save_pretrained(best)
    print(f"TrOCR сохранён → {best}")
    return best


@torch.no_grad()
def _infer_trocr(cfg: ModelConfig, ckpt_dir: str, paths: list, batch: int = 16) -> list:
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    from PIL import Image
    proc = TrOCRProcessor.from_pretrained(ckpt_dir)
    model = VisionEncoderDecoderModel.from_pretrained(ckpt_dir, low_cpu_mem_usage=False)
    model.encoder.pooler = None
    # фикс meta-тензоров (нужен при загрузке оригинального trocr-*-stage1)
    for m in model.modules():
        if hasattr(m, "_float_tensor"):
            m._float_tensor = torch.zeros(1)
        w = getattr(m, "weights", None)
        if torch.is_tensor(w) and w.is_meta:
            n, d = w.shape
            m.weights = m.get_embedding(n, d, getattr(m, "padding_idx", None))
    model.to("cuda")
    for m in model.modules():
        w = getattr(m, "weights", None)
        if torch.is_tensor(w) and w.device.type != "cuda":
            m.weights = w.to("cuda")
    model.eval()
    preds = []
    for i in range(0, len(paths), batch):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i + batch]]
        pv = proc(images=imgs, return_tensors="pt").pixel_values.to("cuda")
        out = model.generate(pv, max_new_tokens=cfg.maxlen, num_beams=4, length_penalty=1.0)
        preds += [s.strip() for s in proc.batch_decode(out, skip_special_tokens=True)]
    del model
    torch.cuda.empty_cache()
    return preds

In [16]:
# =============================================================================
# 6. Qwen-VL (Qwen2-VL / Qwen2.5-VL / Qwen3-VL) — обучение и оценка (QLoRA)
# =============================================================================

SYS_PROMPT = ("You are an expert runologist OCR system. You read runic inscriptions "
              "and output ONLY their transliteration in the Rundata convention, "
              "with no explanation.")
USR_PROMPT = "Transliterate the runic inscription in this image."


def _vlm_class():
    """Единый класс-загрузчик для всех Qwen-VL (с запасным вариантом)."""
    try:
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText
    except Exception:
        from transformers import AutoModelForVision2Seq
        return AutoModelForVision2Seq


def _qwen_bnb():
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=torch.float16,
                              llm_int8_skip_modules=["visual", "lm_head"])


def _qwen_msgs(image, max_pixels, answer=None):
    m = [{"role": "system", "content": SYS_PROMPT},
         {"role": "user", "content": [
             {"type": "image", "image": image, "max_pixels": max_pixels},
             {"type": "text", "text": USR_PROMPT}]}]
    if answer is not None:
        m.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return m


def _image_token_id(model, proc):
    tid = getattr(getattr(model, "config", None), "image_token_id", None)
    if tid is None:
        try:
            tid = proc.tokenizer.convert_tokens_to_ids("<|image_pad|>")
        except Exception:
            tid = -1
    return tid


def _make_qwen_collator(proc, cfg, image_token_id):
    from qwen_vl_utils import process_vision_info

    def collate(batch):
        full = [_qwen_msgs(b["image"], cfg.max_pixels, b["answer"]) for b in batch]
        texts = [proc.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                 for m in full]
        imgs, _ = process_vision_info(full)
        enc = proc(text=texts, images=imgs, padding=True, return_tensors="pt")
        labels = enc["input_ids"].clone()
        labels[labels == proc.tokenizer.pad_token_id] = -100
        if image_token_id is not None and image_token_id >= 0:
            labels[labels == image_token_id] = -100
        # маскируем промпт — учим только ответ ассистента
        for i, b in enumerate(batch):
            pm = _qwen_msgs(b["image"], cfg.max_pixels, None)
            pt = proc.apply_chat_template(pm, tokenize=False, add_generation_prompt=True)
            pim, _ = process_vision_info(pm)
            plen = proc(text=[pt], images=pim, return_tensors="pt")["input_ids"].shape[1]
            labels[i, :plen] = -100
        enc["labels"] = labels
        return enc

    return collate


class _QwenDataset(torch.utils.data.Dataset):
    def __init__(self, df, target):
        self.df = df.reset_index(drop=True)
        self.target = target

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {"image": r["path"], "answer": str(r[self.target])}


def train_qwen(cfg: ModelConfig, df: pd.DataFrame) -> str:
    from transformers import AutoProcessor, Trainer, TrainingArguments, EarlyStoppingCallback
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    set_seed(CFG.SEED)

    proc = AutoProcessor.from_pretrained(cfg.model_id, max_pixels=cfg.max_pixels)
    proc.tokenizer.padding_side = "right"

    ModelCls = _vlm_class()
    model = ModelCls.from_pretrained(
        cfg.model_id, quantization_config=_qwen_bnb() if cfg.quantize_4bit else None,
        torch_dtype=torch.float16, device_map="auto")
    if cfg.quantize_4bit:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False

    if cfg.use_lora:
        model = get_peft_model(model, LoraConfig(
            r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
            bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"]))
        model.print_trainable_parameters()
    model.is_parallelizable = False

    image_token_id = _image_token_id(model, proc)
    collate = _make_qwen_collator(proc, cfg, image_token_id)

    cbs = ([EarlyStoppingCallback(early_stopping_patience=CFG.EARLY_STOPPING_PATIENCE)]
           if CFG.EARLY_STOPPING_PATIENCE > 0 else [])

    args = TrainingArguments(
        output_dir=cfg.out_dir, per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size, gradient_accumulation_steps=cfg.grad_accum,
        num_train_epochs=cfg.epochs, learning_rate=cfg.lr, warmup_ratio=0.05, fp16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, save_total_limit=2,
        logging_steps=20, dataloader_num_workers=2, remove_unused_columns=False,
        report_to="none", optim="paged_adamw_8bit",
    )
    trainer = Trainer(model=model, args=args, data_collator=collate,
                      train_dataset=_QwenDataset(df[df.split == "train"], cfg.target),
                      eval_dataset=_QwenDataset(df[df.split == "val"], cfg.target),
                      callbacks=cbs)
    trainer.train()

    adapter = str(Path(cfg.out_dir) / "lora-adapter")
    model.save_pretrained(adapter)
    proc.save_pretrained(adapter)
    print(f"адаптер сохранён → {adapter}")
    del model, trainer
    torch.cuda.empty_cache()
    return adapter


@torch.no_grad()
def _infer_qwen(cfg: ModelConfig, adapter_dir: str, paths: list) -> list:
    from transformers import AutoProcessor
    from peft import PeftModel
    from qwen_vl_utils import process_vision_info

    proc = AutoProcessor.from_pretrained(adapter_dir, max_pixels=cfg.max_pixels)
    ModelCls = _vlm_class()
    base = ModelCls.from_pretrained(
        cfg.model_id, quantization_config=_qwen_bnb() if cfg.quantize_4bit else None,
        torch_dtype=torch.float16, device_map="auto")
    if os.path.exists(os.path.join(adapter_dir, "adapter_config.json")):
        model = PeftModel.from_pretrained(base, adapter_dir).eval()
    else:
        model = base.eval()

    preds = []
    for p in paths:
        msgs = _qwen_msgs(str(p), cfg.max_pixels, None)
        text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        imgs, _ = process_vision_info(msgs)
        inp = proc(text=[text], images=imgs, return_tensors="pt").to(model.device)
        out = model.generate(**inp, max_new_tokens=cfg.gen_max_new_tokens, do_sample=False)
        dec = proc.batch_decode(out[:, inp["input_ids"].shape[1]:],
                                skip_special_tokens=True)[0]
        preds.append(dec.strip())
    del model, base
    torch.cuda.empty_cache()
    return preds

In [17]:
# =============================================================================
# 7. ДИСПЕТЧЕРЫ ОБУЧЕНИЯ / ОЦЕНКИ
# =============================================================================

def _checkpoint_path(cfg: ModelConfig) -> str:
    return str(Path(cfg.out_dir) / ("best" if cfg.family == "trocr" else "lora-adapter"))


def predict(cfg: ModelConfig, paths: list, ckpt_dir: str = None) -> list:
    ckpt = ckpt_dir or _checkpoint_path(cfg)
    if cfg.family == "trocr":
        return _infer_trocr(cfg, ckpt, paths)
    return _infer_qwen(cfg, ckpt, paths)


def prepare_data(gold_df: pd.DataFrame = None) -> tuple:
    """Подготовить синтетику (со сплитами) и gold set. Возвращает (synth_df, gold_df)."""
    if gold_df is None:
        gold_df = load_gold_set()
    gold_lexicon = set(gold_df["gt"].astype(str))            # для исключения утечки по слову
    # gold set хранит транслитерацию, не руны; чтобы исключать по рунам — сверяем translit
    synth_root = unzip_if_needed(CFG.SYNTH_ZIP, CFG.SYNTH_ROOT)
    synth = parse_synth_labels(synth_root)
    # исключаем синт-строки, чья транслитерация совпала со словом из gold set
    if CFG.EXCLUDE_GOLD_LEXICON:
        n0 = len(synth)
        synth = synth[~synth["translit"].isin(gold_lexicon)].reset_index(drop=True)
        if n0 - len(synth):
            print(f"исключено по лексикону gold set (translit): {n0 - len(synth)} строк")
    # сплиты: канонические из CSV (если заданы) либо по уникальному слову
    synth = make_splits(synth, gold_lexicon=set())
    return synth, gold_df


def run_train(name: str, synth_df: pd.DataFrame = None, gold_df: pd.DataFrame = None) -> str:
    cfg = EXPERIMENTS[name]
    print(f"\n===== ОБУЧЕНИЕ {name} ({cfg.model_id}) =====\n{cfg.note}")
    if synth_df is None:
        synth_df, gold_df = prepare_data(gold_df)
    ckpt = train_trocr(cfg, synth_df) if cfg.family == "trocr" else train_qwen(cfg, synth_df)
    archive_run_outputs(name, best_subdir=("best" if cfg.family == "trocr" else "lora-adapter"))
    return ckpt


def run_eval(name: str, gold_df: pd.DataFrame = None, synth_df: pd.DataFrame = None,
             ckpt_dir: str = None, save: bool = True) -> dict:
    """
    Оценить одну модель: CER на синт-val и на gold set (с бутстрэп-ДИ),
    WER/NED/SeqAcc, по-срезовый CER (если есть столбцы), анализ ошибок.
    Возвращает строку результатов для сводной таблицы.
    """
    cfg = EXPERIMENTS[name]
    if gold_df is None or synth_df is None:
        synth_df, gold_df = prepare_data(gold_df)
    Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)

    # --- синтетическая валидация (подвыборка held-out) ---
    eval_split = CFG.SYNTH_EVAL_SPLIT
    val = synth_df[synth_df.split == eval_split]
    if len(val) == 0:                       # запрошенного сплита нет → откат на val
        print(f"[i] сплит '{eval_split}' пуст; синт-оценка считается на 'val'")
        val = synth_df[synth_df.split == "val"]
    if len(val) > CFG.SYNTH_VAL_EVAL_MAX:
        val = val.sample(CFG.SYNTH_VAL_EVAL_MAX, random_state=CFG.SEED)
    synth_preds = predict(cfg, val["path"].tolist(), ckpt_dir)
    synth_m = compute_metrics(val[cfg.target].astype(str).tolist(), synth_preds)

    # --- gold set ---
    gold_preds = predict(cfg, gold_df["path"].tolist(), ckpt_dir)
    gold_refs = gold_df["gt"].astype(str).tolist()
    gold_m = compute_metrics(gold_refs, gold_preds)
    point, med, lo, hi = bootstrap_cer(gold_m["_char_dist"], gold_m["_char_len"])

    # --- по-срезовый CER ---
    slices = {}
    for col in ("system", "material", "preservation"):
        if col in gold_df.columns:
            for val_name, sub in gold_df.groupby(col):
                m = compute_metrics(sub["gt"].astype(str).tolist(),
                                    [gold_preds[i] for i in sub.index])
                slices[f"{col}:{val_name}"] = round(100 * m["CER"], 2)

    # --- анализ ошибок ---
    try:
        ea = error_analysis(gold_refs, gold_preds)
        save_confusion(ea["confusion"], str(Path(cfg.out_dir) / "confusion.csv"))
    except Exception as e:
        print(f"[i] анализ ошибок пропущен ({e}); установите rapidfuzz")
        ea = None

    row = {
        "model": name, "model_id": cfg.model_id, "params": cfg.params,
        "CER_synth_val": round(100 * synth_m["CER"], 2),
        "CER_gold": round(100 * point, 2),
        "CER_gold_ci": f"({100*lo:.1f}–{100*hi:.1f})",
        "WER_gold": round(100 * gold_m["WER"], 2),
        "NED_gold": round(100 * gold_m["NED"], 2),
        "SeqAcc_gold": round(100 * gold_m["SeqAcc"], 2),
        "dCER": round(100 * (point - synth_m["CER"]), 2),
        "n_gold": gold_m["n"],
        "_gd": gold_m["_char_dist"], "_gl": gold_m["_char_len"],  # для парного бутстрэпа
        "slices": slices,
        "errors": (None if ea is None else
                   {"S%": round(100*ea["S_frac"],1), "D%": round(100*ea["D_frac"],1),
                    "I%": round(100*ea["I_frac"],1), "top_subs": ea["top_subs"]}),
    }

    if save:
        pd.DataFrame({"file": gold_df["file_name"], "gt": gold_refs,
                      "pred": gold_preds}).to_csv(
            Path(cfg.out_dir) / "gold_predictions.csv", index=False, encoding="utf-8")
        public = {k: v for k, v in row.items() if not k.startswith("_")}
        with open(Path(cfg.out_dir) / "metrics.json", "w", encoding="utf-8") as f:
            json.dump(public, f, ensure_ascii=False, indent=2, default=str)

    print(f"[{name:14}] CER_synth={row['CER_synth_val']:5}%  "
          f"CER_gold={row['CER_gold']:5}% {row['CER_gold_ci']}  "
          f"WER={row['WER_gold']:5}%  ΔCER={row['dCER']:+5}  n={row['n_gold']}")

    # --- вывод gt / pred по каждому примеру gold set ---
    gp = pd.DataFrame({"gt": gold_refs, "pred": gold_preds})
    gp["ok"] = gp.apply(lambda r: "✓" if norm_cer(r["gt"]) == norm_cer(r["pred"]) else "", axis=1)
    with pd.option_context("display.max_colwidth", 60, "display.max_rows", 200):
        print(gp.to_string(index=True))

    return row


def run_all_eval(names: list = None, gold_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Оценить все указанные модели и собрать сводную таблицу (как tab:exp-main).
    Дополнительно: парный бутстрэп ΔCER между двумя лучшими по gold-CER.
    """
    names = names or list(EXPERIMENTS.keys())
    synth_df, gold_df = prepare_data(gold_df)
    rows = [run_eval(n, gold_df=gold_df, synth_df=synth_df) for n in names]

    cols = ["model", "params", "CER_synth_val", "CER_gold", "CER_gold_ci",
            "WER_gold", "NED_gold", "SeqAcc_gold", "dCER"]
    table = pd.DataFrame([{c: r[c] for c in cols} for r in rows])
    table = table.sort_values("CER_gold").reset_index(drop=True)
    print("\n===== СВОДНАЯ ТАБЛИЦА (обучение на синтетике, оценка на gold set) =====")
    print(table.to_string(index=False))

    if len(rows) >= 2:
        by_cer = sorted(rows, key=lambda r: r["CER_gold"])
        a, b = by_cer[0], by_cer[1]
        pb = paired_bootstrap_delta(a["_gd"], a["_gl"], b["_gd"], b["_gl"])
        sign = "значима" if pb["significant"] else "НЕ значима"
        print(f"\nПарный бутстрэп ΔCER ({a['model']} − {b['model']}): "
              f"медиана {100*pb['delta_median']:+.2f} п.п., "
              f"95% ДИ [{100*pb['ci_low']:+.2f}; {100*pb['ci_high']:+.2f}] → разность {sign}")

    Path(CFG.OUT_ROOT).mkdir(parents=True, exist_ok=True)
    table.to_csv(Path(CFG.OUT_ROOT) / "comparison_main.csv", index=False, encoding="utf-8")
    return table

In [18]:
# =============================================================================
# 8. АБЛЯЦИЯ РЕЖИМА СИНТЕЗА
# =============================================================================

def run_ablation(name: str, gold_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Абляция режима синтеза для одной (лучшей) модели:
    обучение на {только inpainting | только ControlNet | смесь} → CER на gold set.

    Требует mode-тегированной синтетики (раскладка по подпапкам, см. CFG.ABLATION_MODES).
    Если режимы не размечены — выводит инструкцию и прекращает.
    """
    cfg = EXPERIMENTS[name]
    if gold_df is None:
        gold_df = load_gold_set()
    gold_lexicon = set(gold_df["gt"].astype(str))
    synth_root = unzip_if_needed(CFG.SYNTH_ZIP, CFG.SYNTH_ROOT)
    full = parse_synth_labels(synth_root)
    full = full[~full["translit"].isin(gold_lexicon)].reset_index(drop=True)

    if full["mode"].isna().all():
        print("[!] режимы синтеза не размечены. Для абляции разложите изображения по подпапкам:\n"
              f"      {CFG.SYNTH_ROOT}/inpainting/*.png\n"
              f"      {CFG.SYNTH_ROOT}/controlnet/*.png\n"
              "    (режим «смесь» = обе папки вместе).")
        return pd.DataFrame()

    regimes = {
        "inpainting": full[full["mode"] == "inpainting"],
        "controlnet": full[full["mode"] == "controlnet"],
        "mixed": full,
    }
    results = []
    base_out = CFG.OUT_ROOT
    for reg, sub in regimes.items():
        if len(sub) == 0:
            print(f"[i] режим '{reg}' пуст — пропуск")
            continue
        sub = make_splits(sub, gold_lexicon=set())
        # обучаем во временную подпапку, чтобы не затирать основной чекпойнт
        CFG.OUT_ROOT = str(Path(base_out) / f"ablation_{reg}")
        run_train(name, synth_df=sub, gold_df=gold_df)
        row = run_eval(name, gold_df=gold_df, synth_df=sub, save=True)
        results.append({"regime": reg, "CER_gold": row["CER_gold"],
                        "CER_gold_ci": row["CER_gold_ci"], "n_train": int((sub.split == "train").sum())})
    CFG.OUT_ROOT = base_out

    tab = pd.DataFrame(results)
    if len(tab):
        mixed_cer = tab.loc[tab.regime == "mixed", "CER_gold"]
        if len(mixed_cer):
            tab["delta_to_mixed"] = (tab["CER_gold"] - float(mixed_cer.iloc[0])).round(2)
        print("\n===== АБЛЯЦИЯ РЕЖИМА СИНТЕЗА =====")
        print(tab.to_string(index=False))
        tab.to_csv(Path(base_out) / "ablation_synth.csv", index=False, encoding="utf-8")
    return tab

# # =============================================================================
# # 9. CLI
# # =============================================================================

# def _apply_path_overrides(a) -> None:
#     if a.synth_zip:   CFG.SYNTH_ZIP = a.synth_zip
#     if a.gold_source: CFG.GOLD_SOURCE = a.gold_source
#     if a.out_root:    CFG.OUT_ROOT = a.out_root


# def main():
#     ap = argparse.ArgumentParser(description="Стенд экспериментов ВКР (руническое OCR)")
#     ap.add_argument("--action", required=True,
#                     choices=["train", "eval", "eval-all", "ablation", "list"])
#     ap.add_argument("--model", help="ключ из EXPERIMENTS")
#     ap.add_argument("--models", nargs="+", help="несколько ключей для eval-all")
#     ap.add_argument("--synth-zip"); ap.add_argument("--gold-source"); ap.add_argument("--out-root")
#     a = ap.parse_args()
#     _apply_path_overrides(a)

#     if a.action == "list":
#         for k, c in EXPERIMENTS.items():
#             print(f"  {k:14} {c.model_id:34} [{c.params:>10}]  {c.note}")
#     elif a.action == "train":
#         run_train(a.model)
#     elif a.action == "eval":
#         run_eval(a.model)
#     elif a.action == "eval-all":
#         run_all_eval(a.models)
#     elif a.action == "ablation":
#         run_ablation(a.model)


# if __name__ == "__main__":
#     main()

# run_eval("trocr-base", ckpt_dir=EXPERIMENTS["trocr-base"].model_id)




# =============================================================================
# Kaggle usage
# =============================================================================
# CFG.SYNTH_ZIP = "/kaggle/input/datasets/zhopa228/synth-final/synth_extracted_final"
# CFG.GOLD_SOURCE = "/kaggle/input/datasets/zhopa228/val-dataset/val_dataset"
# CFG.OUT_ROOT = "/kaggle/working/runic_runs"
# CFG.ARCHIVE_ROOT = "/kaggle/working/runic_archives"
#
# synth_df, gold_df = prepare_data()
# run_train("trocr-base", synth_df=synth_df, gold_df=gold_df)
# run_eval("trocr-base", gold_df=gold_df, synth_df=synth_df)
# run_all_eval(["trocr-base", "qwen2vl-2b", "qwen3vl-4b"], gold_df=gold_df)

In [19]:
# run_eval(
#     "qwen2vl-2b",
#     ckpt_dir="Qwen/Qwen2-VL-2B-Instruct"
# )

In [20]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# run_train("qwen2vl-2b")

In [21]:
# run_eval("qwen2vl-2b")

In [22]:
run_train("qwen25vl-7b")


===== ОБУЧЕНИЕ qwen25vl-7b (Qwen/Qwen2.5-VL-7B-Instruct) =====
один T4: на грани; надёжнее Kaggle 2×T4 / Colab A100
gold set: строк=113 | csv=/kaggle/input/datasets/zhopa228/val-dataset/val_dataset/val_dataset/real_corpus.csv
синтетика: строк=4647 | сплиты={'train': 4432, 'val': 215}


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

trainable params: 47,589,376 || all params: 8,339,756,032 || trainable%: 0.5706


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
1,0.504118,0.582116
2,0.248700,0.440774
3,0.084079,0.444597


адаптер сохранён → /kaggle/working/runic_runs/qwen25vl-7b/lora-adapter
[zip] run → /kaggle/working/runic_archives/runs/qwen25vl-7b.zip
[zip] best → /kaggle/working/runic_archives/best/qwen25vl-7b_best.zip


'/kaggle/working/runic_runs/qwen25vl-7b/lora-adapter'

In [23]:
run_eval("qwen25vl-7b")

gold set: строк=113 | csv=/kaggle/input/datasets/zhopa228/val-dataset/val_dataset/val_dataset/real_corpus.csv
синтетика: строк=4647 | сплиты={'train': 4432, 'val': 215}


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

[i] анализ ошибок пропущен (No module named 'rapidfuzz'); установите rapidfuzz
[qwen25vl-7b   ] CER_synth=12.03%  CER_gold=54.27% (48.0–61.3)  WER=81.18%  ΔCER=+42.24  n=113
                                         gt                               pred ok
0                              saRþatbarutR                              htbhu   
1                            utiaRweladaude                              hmweï   
2                              haeramalausR                                alu   
3                            inarunaRarageu                           hun hmhl   
4                           falahakhaiderag                              hhRhi   
5                             haidRrunoronu                         hidRlnorso   
6                                uþarabasba                           uþarhasb   
7                                uþarabasba                         uþhRhbajbh   
8                              niuhagestumR                       hiunagmetïmR   
9     

{'model': 'qwen25vl-7b',
 'model_id': 'Qwen/Qwen2.5-VL-7B-Instruct',
 'params': '~7 млрд',
 'CER_synth_val': 12.03,
 'CER_gold': 54.27,
 'CER_gold_ci': '(48.0–61.3)',
 'WER_gold': 81.18,
 'NED_gold': 59.37,
 'SeqAcc_gold': 8.85,
 'dCER': 42.24,
 'n_gold': 113,
 '_gd': array([ 9, 12,  9, 12, 12,  6,  3,  5,  5,  9,  8,  7,  9,  6,  8,  5,  7,
         3,  3,  5,  8,  9, 10,  7,  3, 10,  7,  3,  0,  4,  3,  9,  2,  3,
         1,  6,  1,  0,  4,  0,  1,  0,  0,  1,  0,  1,  7,  3,  4,  1, 10,
         5,  4, 17,  6,  3,  7,  2,  5,  5,  3,  2,  2,  5, 10,  2,  7,  3,
         4,  4,  1,  0,  7,  0,  7,  4,  6,  6,  3,  8,  3,  3,  3,  0,  4,
         3,  4,  3,  4,  5,  3,  5,  4,  7,  3,  4,  2,  1,  3,  0,  2,  1,
         6,  3,  7,  5,  3,  4,  7,  4,  2,  6,  2]),
 '_gl': array([12, 14, 12, 14, 15, 13, 10, 10, 12, 11, 14, 12, 14,  6, 10,  8, 11,
        10,  4, 12, 13, 10, 10, 15,  3, 13, 12,  7,  5,  4,  3, 11,  3,  8,
         5,  7,  9,  3,  6,  2,  4,  8,  5,  3,  3,  3,  7,  3,

In [25]:
synth_df, gold_df = prepare_data()

gold set: строк=113 | csv=/kaggle/input/datasets/zhopa228/val-dataset/val_dataset/val_dataset/real_corpus.csv
синтетика: строк=4647 | сплиты={'train': 4432, 'val': 215}


In [28]:
gold_df

,file_name,translit,translit_raw,runic_approx,n_chars,source,split,had_word_divider,comment,gt,path
0,93_48_1str.png,saRþatbarutR,sᴀzþᴀtbᴀrutz,ᛊᚨᛉᚦᚨᛏᛒᚨᚱᚢᛏᛉ,12,real,real_test,False,NaN,saRþatbarutR,/kaggle/input/datasets/zhopa228/val-dataset/va...
1,93_48_2str.png,utiaRweladaude,utiᴀzwelᴀdᴀude,ᚢᛏᛁᚨᛉᚹᛖᛚᚨᛞᚨᚢᛞᛖ,14,real,real_test,False,NaN,utiaRweladaude,/kaggle/input/datasets/zhopa228/val-dataset/va...
2,93_48_3str.png,haeramalausR,hᴀerᴀmᴀlᴀusz,ᚺᚨᛖᚱᚨᛗᚨᛚᚨᚢᛊᛉ,12,real,real_test,False,NaN,haeramalausR,/kaggle/input/datasets/zhopa228/val-dataset/va...
3,93_48_4str.png,inarunaRarageu,inᴀrunazᴀrᴀgeu,ᛁᚾᚨᚱᚢᚾᚨᛉᚨᚱᚨᚷᛖᚢ,14,real,real_test,False,NaN,inarunaRarageu,/kaggle/input/datasets/zhopa228/val-dataset/va...
4,93_48_5str.png,falahakhaiderag,fᴀlᴀhᴀkhᴀiderᴀg,ᚠᚨᛚᚨᚺᚨᚲᚺᚨᛁᛞᛖᚱᚨᚷ,15,real,real_test,False,NaN,falahakhaiderag,/kaggle/input/datasets/zhopa228/val-dataset/va...
...,...,...,...,...,...,...,...,...,...,...,...
108,5810_0_kuþmutr-karþi.png,kuþmutr karþi,NaN,NaN,12,synth,train,True,NaN,kuþmutr karþi,/kaggle/input/datasets/zhopa228/val-dataset/va...
109,5810_0_kuþmutr.png,kuþmutr,NaN,NaN,7,synth,train,False,NaN,kuþmutr,/kaggle/input/datasets/zhopa228/val-dataset/va...
110,5810_0_þusi-eftiR-urmar.png,þusi eftiR urmar,NaN,NaN,14,synth,train,True,NaN,þusi eftiR urmar,/kaggle/input/datasets/zhopa228/val-dataset/va...
111,5810_0_þusi-eftiR.png,þusi eftiR,NaN,NaN,9,synth,train,True,NaN,þusi eftiR,/kaggle/input/datasets/zhopa228/val-dataset/va...


In [27]:
msgs = _qwen_msgs(
    str(path),
    cfg.max_pixels,
    None
)

from qwen_vl_utils import process_vision_info

imgs, _ = process_vision_info(msgs)

print(type(imgs[0]))
print(imgs[0].size)

NameError: name 'path' is not defined